# Global Climate Change Data Analysis

Reconstructed from the CS22A final project slides.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, f_oneway, pearsonr

# Data source used in the slides
DATA_URL = "https://raw.githubusercontent.com/csbfx/cs22a/main/global_climate.csv"

In [ ]:
# Load dataset
climate_df = pd.read_csv(DATA_URL)
print(climate_df)

In [ ]:
# Data cleaning / inspection
print("Unique Regions:", climate_df["Region"].unique())
print("Unique Countries:", climate_df["Country"].nunique())
print(climate_df.isnull().sum())

# Drop rows with missing values
climate_df.dropna(inplace=True)

# Create a dictionary of dataframes for each region
regions = climate_df["Region"].unique()
region_data = {
    region: climate_df[climate_df["Region"] == region]
    for region in regions
}

In [ ]:
# Count the number of unique cities per region
city_counts = climate_df.groupby("Region")["City"].nunique()

plt.figure(figsize=(10, 6))
city_counts.sort_values().plot(kind="bar", color="skyblue")
plt.title("Number of Cities per Region")
plt.xlabel("Region")
plt.ylabel("Number of Cities")
plt.xticks(rotation=45)
plt.grid()
plt.show()

In [ ]:
# Summary statistics for each region
region_stats = {}

for region, df in region_data.items():
    df_clean = df.dropna(subset=["AvgTemperature"])

    mean_temp = df_clean["AvgTemperature"].mean()
    median_temp = df_clean["AvgTemperature"].median()
    mode_series = df_clean["AvgTemperature"].mode()
    mode_temp = mode_series.iloc[0] if not mode_series.empty else None

    region_stats[region] = {
        "Mean": mean_temp,
        "Median": median_temp,
        "Mode": mode_temp,
        "City Count": df_clean["City"].nunique(),
        "Data Count": len(df_clean),
    }

    print(f"Region: {region}")
    print(f"  Mean Temperature: {mean_temp:.2f} °F")
    print(f"  Median Temperature: {median_temp:.2f} °F")
    if mode_temp is not None:
        print(f"  Mode Temperature: {mode_temp:.2f} °F")
    else:
        print("  Mode Temperature: None")
    print(f"  Cities Represented: {df_clean['City'].nunique()}")
    print(f"  Data Points: {len(df_clean)}\n")

In [ ]:
# Compare average temperatures before and after 2010 with t-tests
before_2010 = 1995
after_2010 = 2010

for region, df in region_data.items():
    df_clean = df.dropna(subset=["AvgTemperature"])

    before_2010_data = df_clean[df_clean["Year"] < after_2010]["AvgTemperature"]
    after_2010_data = df_clean[df_clean["Year"] >= after_2010]["AvgTemperature"]

    t_stat, p_value = ttest_ind(before_2010_data, after_2010_data)

    print(f"Region: {region}")
    print(f"T-Statistic: {t_stat:.2f}")
    print(f"P-Value: {p_value:.4f}")
    print("-" * 40)

In [ ]:
# One-way ANOVA across regions
region_temps = [
    df["AvgTemperature"].dropna()
    for region, df in region_data.items()
]

f_stat, p_value = f_oneway(*region_temps)

print("One-Way ANOVA Results:")
print(f"F-Statistic: {f_stat:.2f}")
print(f"P-Value: {p_value:.4f}")

In [ ]:
# Visualize annual temperature trends by region
for region, df in region_data.items():
    df_clean = df.dropna(subset=["AvgTemperature"])

    annual_means = df_clean.groupby("Year")["AvgTemperature"].mean()

    plt.figure(figsize=(10, 6))
    plt.plot(
        annual_means.index,
        annual_means.values,
        marker="o",
        linestyle="-",
        color="blue",
    )
    plt.title(f"Annual Temperature Trends - {region}")
    plt.xlabel("Year")
    plt.ylabel("Average Temperature (°F)")
    plt.grid()
    plt.show()

In [ ]:
# Pearson correlation between year and average temperature by region
for region, df in region_data.items():
    df_clean = df.dropna(subset=["AvgTemperature"])

    annual_means = df_clean.groupby("Year")["AvgTemperature"].mean().reset_index()

    corr, p_value = pearsonr(annual_means["Year"], annual_means["AvgTemperature"])

    print(f"Region: {region}")
    print(f"Correlation Coefficient: {corr:.2f}")
    print(f"P-Value: {p_value:.4f}")
    print("-" * 40)

In [ ]:
# Scatter plots with trendlines for each region
for region, df in region_data.items():
    df_clean = df.dropna(subset=["AvgTemperature"])

    annual_means = df_clean.groupby("Year")["AvgTemperature"].mean().reset_index()

    plt.figure(figsize=(10, 6))
    sns.regplot(
        x="Year",
        y="AvgTemperature",
        data=annual_means,
        color="blue",
        marker="o",
        line_kws={"color": "green"},
    )
    plt.title(f"Correlation Between Year and AvgTemperature - {region}", fontsize=14)
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Temperature (°F)", fontsize=12)
    plt.grid()
    plt.show()

In [ ]:
# Count the number of records each year
print(climate_df["Year"].value_counts())